# Module 10: Prediction and evaluation

**Goal:** build models that predict well on **new** data, and evaluate them honestly. This module is about *generalisation*: the gap between how a model does on the
data it learned from and how it will do in the real world.

**You will learn:** train/test splits · baselines · cross-validation · overfitting and regularisation · data leakage and pipelines ·
logistic regression · classification metrics (precision, recall, ROC/AUC) · trees and forests · time-ordered data

**Maths in this module:** loss functions · the bias–variance trade-off · odds, log-odds and the sigmoid · confusion-matrix formulas · the ridge penalty

> 🎯 **Explaining** (Module 9) asks "what is the relationship, and how sure are we?". **Predicting** asks "how well will this work on cases I haven't seen?"
> The second question can *only* be answered with data the model hasn't touched.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid", palette="colorblind")
students = pd.read_csv(DATA / "students.csv")
feats = ["study_hours_week", "sleep_hours", "attendance_pct"]
X, y = students[feats], students["exam_score"]

## 1. Overfitting: the central problem

A flexible enough model can memorise noise. We'll fit polynomials of increasing degree to 30 noisy points from a sine wave, keeping some points **hidden** to test on.

In [ ]:
rng = np.random.default_rng(10)
xs = np.sort(rng.uniform(0, 1, 30))
ys = np.sin(2 * np.pi * xs) + rng.normal(0, 0.3, 30)
x_tr, x_te, y_tr, y_te = train_test_split(xs.reshape(-1, 1), ys, test_size=0.4, random_state=1)

degrees = [1, 3, 5, 9, 12]
rows = []
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(x_tr, y_tr)
    rows.append((d, mean_squared_error(y_tr, m.predict(x_tr)), mean_squared_error(y_te, m.predict(x_te))))
curve = pd.DataFrame(rows, columns=["degree", "train_mse", "test_mse"]).round(3)
curve

In [ ]:
grid = np.linspace(0, 1, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, d in zip(axes, [1, 3, 12]):
    m = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(x_tr, y_tr)
    ax.scatter(x_tr, y_tr, label="train"); ax.scatter(x_te, y_te, marker="x", color="tab:red", label="test")
    ax.plot(grid, m.predict(grid), color="black"); ax.set_ylim(-2, 2)
    ax.set_title(f"degree {d}: " + {1: "too simple (underfit)", 3: "about right", 12: "too flexible (overfit)"}[d])
axes[0].legend(); plt.tight_layout(); plt.show()

* **Training error always falls** as the model gets more flexible.
* **Test error falls, then rises**: past the sweet spot the model fits the noise.

This is the **bias–variance trade-off**: a too-simple model has high *bias* (it can't capture the truth), a too-flexible one has high *variance* (its fit swings wildly with each new sample).
The goal is the bottom of the U in **test** error, never the training error.

## 2. Honest evaluation: split, baseline, metrics

**Hold out** a test set that the model never sees during fitting, and use it **once** at the end.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(X_train.shape, X_test.shape)

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)      # always predicts the training mean

def report(name, y_true, y_pred):
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:10s} MAE {mean_absolute_error(y_true, y_pred):5.2f}   RMSE {rmse:5.2f}   R² {r2_score(y_true, y_pred):6.3f}")

report("baseline", y_test, baseline.predict(X_test))
report("linear", y_test, pred)

**Always compare against a baseline** (predict the mean, the majority class, last month's value). A model is only useful if it beats the simplest sensible rule.

| metric | formula | in words |
|---|---|---|
| **MAE** | $\frac1n\sum\lvert y_i-\hat y_i\rvert$ | typical miss, in $y$-units; robust to outliers |
| **RMSE** | $\sqrt{\frac1n\sum(y_i-\hat y_i)^2}$ | typical miss, punishes big misses |
| **$R^2$** | $1-\text{SSE}/\text{SST}$ | share of variance explained vs the "predict the mean" baseline (can be **negative** on test data) |

## 3. Cross-validation: don't trust a single split

One random split can be lucky or unlucky. **$k$-fold cross-validation** splits the data into $k$ parts, trains on $k-1$ and tests on the remaining one, rotating through all parts, giving $k$ scores.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(LinearRegression(), X, y, cv=cv, scoring="r2")
print("R² per fold:", scores.round(3))
print(f"mean {scores.mean():.3f}  ±  {scores.std():.3f}")

The folds disagree noticeably (the spread is large with only 200 rows). That is information too: **report the mean *and* the spread**, and be wary of reading much into a single split.

CV also exposes overfitting. Add lots of columns of **pure noise** and watch training fit improve while cross-validated performance collapses:

In [ ]:
for k in [0, 10, 60, 120]:
    rng = np.random.default_rng(0)
    noise = pd.DataFrame(rng.normal(size=(len(X), k)), columns=[f"noise{i}" for i in range(k)])
    Xk = pd.concat([X, noise], axis=1)
    train_r2 = LinearRegression().fit(Xk, y).score(Xk, y)
    cv_r2 = cross_val_score(LinearRegression(), Xk, y, cv=cv, scoring="r2").mean()
    print(f"{k:3d} junk columns:  training R² {train_r2:6.3f}   cross-validated R² {cv_r2:7.3f}")

With 120 junk columns the training $R^2$ is a glowing 0.87, yet on unseen data the model is **worse than predicting the mean** (negative $R^2$).
(With only 10 junk columns and 200 rows the damage is hardly visible; overfitting bites when features are many compared with rows.)

## 4. Data leakage and pipelines

**Leakage** is when information from outside the training data sneaks into the model, making scores look great and then failing in the real world. Two common forms:

1. **Target leakage:** a feature that wouldn't be known at prediction time, or is derived from the answer.

In [ ]:
leaky = X.assign(grade_points=y / 25)                    # a column computed from the exam score itself
print("CV R² with a leaky feature:", cross_val_score(LinearRegression(), leaky, y, cv=cv, scoring="r2").mean().round(3))

An $R^2$ of 1.000 is not a triumph; it's a **red flag**. Ask "would I really have this column when predicting a *new* case?"

2. **Preprocessing leakage:** fitting a scaler (or imputer, encoder…) on the *whole* dataset before splitting lets test-set statistics leak into training.
The cure is a **Pipeline**: all steps are learned from the training fold only, automatically, inside cross-validation.

In [ ]:
pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
print(pipe)
print("CV R² (scaler fitted inside each fold):", cross_val_score(pipe, X, y, cv=cv, scoring="r2").mean().round(3))

## 5. Regularisation: taming flexible models

Instead of limiting the degree, **penalise large coefficients**. **Ridge** regression minimises

$$\sum_i(y_i-\hat y_i)^2\;+\;\alpha\sum_j b_j^2$$

The penalty $\alpha$ is a **hyperparameter** that you choose (not learned from the data directly), using cross-validation *on the training data*. Bigger $\alpha$ shrinks coefficients toward 0 (simpler fit).
Standardise features first so the penalty treats them fairly. (Lasso uses $\alpha\sum\lvert b_j\rvert$ and can drive coefficients exactly to zero.)

In [ ]:
poly_ridge = make_pipeline(PolynomialFeatures(12), StandardScaler(), Ridge())
search = GridSearchCV(poly_ridge, {"ridge__alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1, 10]},
                      cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_squared_error").fit(x_tr, y_tr)

unreg = make_pipeline(PolynomialFeatures(12), StandardScaler(), LinearRegression()).fit(x_tr, y_tr)
print("best alpha chosen by CV:", search.best_params_["ridge__alpha"])
print("test MSE, degree 12 unregularised:", round(mean_squared_error(y_te, unreg.predict(x_te)), 3))
print("test MSE, degree 12 + ridge      :", round(mean_squared_error(y_te, search.predict(x_te)), 3))

The same flexible model, now constrained, predicts far better on the hidden points.

## 6. Classification and logistic regression

Now predict a **category**. Define a **high achiever** as a student scoring 75 or above (about 24% of students).

In [ ]:
high = (y >= 75).astype(int)
print("share of high achievers:", high.mean().round(3))
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, high, test_size=0.25, random_state=0, stratify=high)   # stratify keeps the class mix

A straight line can't output a probability. **Logistic regression** models the *log-odds* as linear and squashes it through the **sigmoid**:

$$\text{odds}=\frac{p}{1-p},\qquad \log\frac{p}{1-p}=b_0+b_1x_1+\dots\qquad\Longleftrightarrow\qquad p=\sigma(z)=\frac{1}{1+e^{-z}}$$

The sigmoid maps any real number $z$ to $(0,1)$. We then **classify** by comparing $p$ with a threshold (0.5 by default).

In [ ]:
z = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.plot(z, 1 / (1 + np.exp(-z))); ax.axhline(0.5, color="gray", linestyle=":")
ax.set_xlabel("z (log-odds)"); ax.set_ylabel("probability p"); ax.set_title("The sigmoid squashes log-odds into probabilities")
plt.show()

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xc_train, yc_train)
proba = clf.predict_proba(Xc_test)[:, 1]          # P(high achiever)
pred_class = clf.predict(Xc_test)                 # proba >= 0.5

coefs = clf[-1].coef_[0]
print(pd.DataFrame({"log-odds per +1 sd": coefs, "odds ratio": np.exp(coefs)}, index=feats).round(2))

Because features are standardised, an **odds ratio** of e.g. 5 means: one standard deviation more of that feature multiplies the odds of being a high achiever by about 5 (others held fixed).

## 7. Evaluating classifiers

Every prediction lands in one of four boxes:

| | predicted **positive** | predicted **negative** |
|---|---|---|
| **actually positive** | true positive (TP) | false negative (FN) |
| **actually negative** | false positive (FP) | true negative (TN) |

In [ ]:
cm = confusion_matrix(yc_test, pred_class)
tn, fp, fn, tp = cm.ravel()
print(cm, f"-> TN={tn} FP={fp} FN={fn} TP={tp}")

fig, ax = plt.subplots(figsize=(3.8, 3.4))
ConfusionMatrixDisplay.from_predictions(yc_test, pred_class, display_labels=["not high", "high"], ax=ax, colorbar=False)
plt.show()

| metric | formula | answers |
|---|---|---|
| **accuracy** | $\dfrac{TP+TN}{\text{all}}$ | how often is it right overall? |
| **precision** | $\dfrac{TP}{TP+FP}$ | when it says "positive", how often is it right? |
| **recall** (sensitivity) | $\dfrac{TP}{TP+FN}$ | of the real positives, how many did it find? |
| **F1** | $\dfrac{2\,\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$ | one number balancing the two |

In [ ]:
print(f"accuracy  {(tp + tn) / cm.sum():.3f}")
print(f"precision {tp / (tp + fp):.3f}   (sklearn: {precision_score(yc_test, pred_class):.3f})")
print(f"recall    {tp / (tp + fn):.3f}   (sklearn: {recall_score(yc_test, pred_class):.3f})")
print(f"F1        {f1_score(yc_test, pred_class):.3f}")

### The accuracy trap

With 24% positives, a "model" that always says *no* is right 76% of the time and finds **zero** positives:

In [ ]:
always_no = DummyClassifier(strategy="most_frequent").fit(Xc_train, yc_train)
p = always_no.predict(Xc_test)
print(f"always 'no': accuracy {(p == yc_test).mean():.2f}, recall {recall_score(yc_test, p):.2f}")

For rare events (fraud, disease, churn) **accuracy is nearly useless**; use recall, precision, F1 or AUC, and compare with the baseline.
Which error is worse depends on the situation: missing a disease (FN) vs. a needless follow-up test (FP). **Choose the metric from the cost of mistakes.**

### Thresholds and the ROC curve
The 0.5 cutoff is a choice. Lowering it catches more positives (higher recall) but raises more false alarms (lower precision):

In [ ]:
rows = []
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pr = (proba >= t).astype(int)
    rows.append((t, pr.sum(), precision_score(yc_test, pr, zero_division=0), recall_score(yc_test, pr)))
pd.DataFrame(rows, columns=["threshold", "flagged", "precision", "recall"]).round(2)

The **ROC curve** plots the true-positive rate (recall) against the false-positive rate for *every* threshold. The **AUC** (area under it) summarises ranking quality:
0.5 = coin-flip, 1.0 = perfect.

In [ ]:
fpr, tpr, _ = roc_curve(yc_test, proba)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(fpr, tpr, label=f"logistic (AUC = {roc_auc_score(yc_test, proba):.2f})")
ax.plot([0, 1], [0, 1], linestyle=":", color="gray", label="coin flip")
ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate (recall)"); ax.legend()
plt.show()

## 8. Other model families (and why "start simple")

**Decision trees** split the data with a sequence of yes/no questions; **random forests** average many trees. They capture curves and interactions automatically but can overfit.
Compare with cross-validated AUC:

In [ ]:
cv_c = KFold(5, shuffle=True, random_state=0)
candidates = {
    "baseline (always no)": DummyClassifier(strategy="most_frequent"),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "tree, depth 3": DecisionTreeClassifier(max_depth=3, random_state=0),
    "tree, unlimited depth": DecisionTreeClassifier(random_state=0),
    "random forest": RandomForestClassifier(n_estimators=200, random_state=0),
}
for name, mdl in candidates.items():
    auc = cross_val_score(mdl, X, high, cv=cv_c, scoring="roc_auc")
    print(f"{name:24s} AUC {auc.mean():.3f} ± {auc.std():.3f}")

deep = DecisionTreeClassifier(random_state=0).fit(Xc_train, yc_train)
print(f"\nunlimited tree: train accuracy {deep.score(Xc_train, yc_train):.2f}  vs  test accuracy {deep.score(Xc_test, yc_test):.2f}   <- memorised the training set")

The humble logistic regression is as good as the forest and far better than a deep tree, and it's **easy to explain**. That's no accident here: the underlying pattern is close to linear.
Reach for complex models when a simple one demonstrably isn't enough. **Complexity must earn its place in cross-validation.**

## 9. Time-ordered data

If rows have an order (days, months), **don't shuffle**. Train on the past, test on the future, as you would in real use. Random splits let the model peek at days right next to those it's tested on.

In [ ]:
rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"]).dropna().sort_values("date").reset_index(drop=True)
cols = ["temp_c", "is_weekend"]

train, test = rides.iloc[:300], rides.iloc[300:]               # chronological split
m = LinearRegression().fit(train[cols], train["rides"])
rmse_time = math.sqrt(mean_squared_error(test["rides"], m.predict(test[cols])))

random_rmse = []
for seed in range(20):
    a, b = train_test_split(rides, test_size=60, random_state=seed)
    mm = LinearRegression().fit(a[cols], a["rides"])
    random_rmse.append(math.sqrt(mean_squared_error(b["rides"], mm.predict(b[cols]))))

print(f"train on first 300 days, test on last {len(test)}: RMSE {rmse_time:.0f} rides")
print(f"20 random splits: RMSE averages {np.mean(random_rmse):.0f}, ranging {min(random_rmse):.0f} to {max(random_rmse):.0f}")

Two honest observations. First, **single splits are noisy** (random-split RMSE swings between about 90 and 140): another reason to repeat evaluations.
Second, here the chronological estimate (109) is in the same ballpark as the random-split average (122), well within the spread of the random splits, because this synthetic data has independent days given the weather.
Real data with trends, seasons or autocorrelation often show a much bigger gap, so the chronological split is the safe, honest default.

## 10. The evaluation checklist

1. Define the **prediction task** and what "good" means in real terms (cost of errors).
2. Make a **baseline**. A model must beat it.
3. **Split before looking** (random, stratified, grouped, or chronological, as the data demand). Put the test set aside.
4. Do all choices (features, hyperparameters, model type) with **cross-validation on the training set**.
5. Use **pipelines** so preprocessing is learned from training data only. Hunt for **leakage** (scores too good to be true).
6. Touch the test set **once**, at the end; report the metric **with its uncertainty**.
7. Explain the model's behaviour (coefficients, importances) and its **limits**: where would it fail?

## Recap

* Training error flatters; **test error tells the truth**. Overfitting = learning noise. Regularisation and simpler models fight it.
* Cross-validation gives a more stable estimate and exposes overfitting. Pipelines prevent preprocessing leakage.
* Logistic regression: linear in **log-odds**, probabilities via the sigmoid. Metrics beyond accuracy: precision, recall, F1, ROC/AUC. Thresholds are a trade-off.
* Beat a baseline, start simple, and report performance honestly.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** (model comparison).